In [1]:
!pip -q install --upgrade pip
!pip -q install timm albumentations opencv-python-headless scikit-learn pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 91.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re
import sys
import json
import random
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import timm

DRIVE_ROOT = Path('/content/drive/MyDrive/ML/Findit-2026')
PROJECT_ROOT = DRIVE_ROOT

DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
SPLIT_DIR = DATA_DIR / 'splits'

EXP_DIR = PROJECT_ROOT / 'experiments'
CKPT_DIR = EXP_DIR / 'checkpoints'
OOF_DIR = EXP_DIR / 'oof_predictions'
LOG_DIR = EXP_DIR / 'run_logs'

OUT_DIR = PROJECT_ROOT / 'output'
SUB_DIR = OUT_DIR / 'submissions'
TEST_PROB_DIR = OUT_DIR / 'test_probabilities'

REPORT_DIR = PROJECT_ROOT / 'reports'
TABLE_DIR = REPORT_DIR / 'tables'
FIG_DIR = REPORT_DIR / 'figures'

for p in [CKPT_DIR, OOF_DIR, LOG_DIR, SUB_DIR, TEST_PROB_DIR, TABLE_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

TRAIN_DIR = RAW_DIR / 'train'
TEST_DIR = RAW_DIR / 'test'
SAMPLE_SUB_PATH = RAW_DIR / 'samplesubmission.csv'
SPLIT_PATH = SPLIT_DIR / 'train_5fold_stratified_group_seed42.csv'
TEST_META_PATH = PROCESSED_DIR / 'clean' / 'test_metadata_with_hash.csv'

required_paths = [TRAIN_DIR, TEST_DIR, SAMPLE_SUB_PATH, SPLIT_PATH, TEST_META_PATH]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError('Path berikut belum ada di Drive:\n- ' + '\n- '.join(missing))

split_df = pd.read_csv(SPLIT_PATH)
test_meta_df = pd.read_csv(TEST_META_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

required_split_cols = {'file_path', 'label', 'fold'}
if not required_split_cols.issubset(set(split_df.columns)):
    raise ValueError(f'split_df missing columns: {required_split_cols - set(split_df.columns)}')

if split_df['fold'].nunique() < 5:
    raise ValueError('Fold unik kurang dari 5, cek file split.')

if len(test_meta_df) != len(sample_sub):
    raise ValueError('Jumlah test metadata tidak sama dengan sample submission.')

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

@dataclass
class CFG:
    seed: int = 42

    # Pilihan model:
    # - 'swinv2_base_window12_192'                  -> kuat untuk texture/detail spoof
    # - 'convnext_base.fb_in22k_ft_in1k'            -> stabil dan robust
    # - 'efficientnetv2_m.in21k_ft_in1k'            -> efisien dan strong baseline
    model_name: str = 'swinv2_base_window12_192'

    img_size: int = 192
    batch_size: int = 12
    num_workers: int = 2
    num_workers_val: int = 2
    num_workers_test: int = 2

    epochs: int = 14
    lr: float = 1.2e-4
    weight_decay: float = 1e-4
    patience: int = 3
    n_splits: int = 5
    run_full_cv: bool = True
    mixed_precision: bool = True

    # Pilihan loss:
    # - 'ce'    : cross entropy standar
    # - 'ce_ls' : cross entropy + label smoothing (umumnya paling stabil)
    # - 'focal' : fokus ke sampel susah, cocok saat class imbalance berat
    loss_name: str = 'ce_ls'
    label_smoothing: float = 0.08
    focal_gamma: float = 2.0
    use_class_weights: bool = True

    use_tta: bool = True
    tta_hflip: bool = True
    tta_light_bc: bool = False
    tta_bc_contrast: float = 1.05
    tta_bc_brightness: float = 0.02

cfg = CFG()
seed_everything(cfg.seed)

def resolve_timm_model_name(requested_model_name: str) -> str:
    pretrained_models = set(timm.list_models(pretrained=True))
    if requested_model_name in pretrained_models:
        return requested_model_name

    base_name = requested_model_name.split('.')[0]
    if base_name in pretrained_models:
        print(f"[INFO] Fallback model_name from {requested_model_name} -> {base_name}")
        return base_name

    candidates = sorted(timm.list_models(f'{base_name}*', pretrained=True))
    if len(candidates) > 0:
        print(f"[INFO] Fallback model_name from {requested_model_name} -> {candidates[0]}")
        return candidates[0]

    raise RuntimeError(
        f"Model pretrained tidak ditemukan untuk '{requested_model_name}'. "
        "Coba model_name lain dari timm.list_models(pretrained=True)."
    )

resolved_model_name = resolve_timm_model_name(cfg.model_name)

# Auto align cfg.img_size if model name includes fixed image size suffix (e.g. _192)
m = re.search(r'_(\d{3})(?:\.|$)', resolved_model_name)
if m is not None:
    fixed_size = int(m.group(1))
    if cfg.img_size != fixed_size:
        print(f"[INFO] Auto-set cfg.img_size from {cfg.img_size} -> {fixed_size} for {resolved_model_name}")
        cfg.img_size = fixed_size

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)
print('DEVICE:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU VRAM: {total_mem_gb:.2f} GB')

print('split_df:', split_df.shape)
print('test_meta_df:', test_meta_df.shape)
print('sample_sub:', sample_sub.shape)
print('fold distribution:\n', split_df['fold'].value_counts().sort_index())
print('resolved_model_name:', resolved_model_name)
print('cfg:', asdict(cfg))
print('READY_FOR_TRAINING: YES')

PROJECT_ROOT: /content/drive/MyDrive/ML/Findit-2026
DATA_DIR: /content/drive/MyDrive/ML/Findit-2026/data


## Training

In [ ]:
RECOMMENDED_RUN_CONFIGS = [
    {
        'name': 'run_swinv2_base_192_ce_ls',
        'seed': 42,
        'model_name': 'swinv2_base_window12_192',
        'img_size': 192,
        'batch_size': 12,
        'num_workers': 2,
        'num_workers_val': 2,
        'num_workers_test': 2,
        'epochs': 14,
        'lr': 1.2e-4,
        'weight_decay': 1e-4,
        'patience': 3,
        'n_splits': 5,
        'run_full_cv': True,
        'mixed_precision': True,
        'loss_name': 'ce_ls',
        'label_smoothing': 0.08,
        'focal_gamma': 2.0,
        'use_class_weights': True,
        'use_tta': True,
        'tta_hflip': True,
        'tta_light_bc': False,
        'tta_bc_contrast': 1.05,
        'tta_bc_brightness': 0.02,
    },
    {
        'name': 'run_convnext_base_224_focal',
        'seed': 42,
        'model_name': 'convnext_base.fb_in22k_ft_in1k',
        'img_size': 224,
        'batch_size': 10,
        'num_workers': 2,
        'num_workers_val': 2,
        'num_workers_test': 2,
        'epochs': 14,
        'lr': 1.2e-4,
        'weight_decay': 1e-4,
        'patience': 3,
        'n_splits': 5,
        'run_full_cv': True,
        'mixed_precision': True,
        'loss_name': 'focal',
        'label_smoothing': 0.08,
        'focal_gamma': 2.0,
        'use_class_weights': True,
        'use_tta': True,
        'tta_hflip': True,
        'tta_light_bc': False,
        'tta_bc_contrast': 1.05,
        'tta_bc_brightness': 0.02,
    },
    {
        'name': 'run_effnetv2m_256_ce_ls',
        'seed': 42,
        'model_name': 'efficientnetv2_m.in21k_ft_in1k',
        'img_size': 256,
        'batch_size': 8,
        'num_workers': 2,
        'num_workers_val': 2,
        'num_workers_test': 2,
        'epochs': 14,
        'lr': 1.0e-4,
        'weight_decay': 1e-4,
        'patience': 3,
        'n_splits': 5,
        'run_full_cv': True,
        'mixed_precision': True,
        'loss_name': 'ce_ls',
        'label_smoothing': 0.08,
        'focal_gamma': 2.0,
        'use_class_weights': True,
        'use_tta': True,
        'tta_hflip': True,
        'tta_light_bc': False,
        'tta_bc_contrast': 1.05,
        'tta_bc_brightness': 0.02,
    },
    {
        'name': 'run_convnext_base_224_ce_ls_no_tta',
        'seed': 42,
        'model_name': 'convnext_base.fb_in22k_ft_in1k',
        'img_size': 224,
        'batch_size': 10,
        'num_workers': 2,
        'num_workers_val': 2,
        'num_workers_test': 2,
        'epochs': 12,
        'lr': 1.5e-4,
        'weight_decay': 1e-4,
        'patience': 3,
        'n_splits': 5,
        'run_full_cv': True,
        'mixed_precision': True,
        'loss_name': 'ce_ls',
        'label_smoothing': 0.08,
        'focal_gamma': 2.0,
        'use_class_weights': True,
        'use_tta': False,
        'tta_hflip': True,
        'tta_light_bc': False,
        'tta_bc_contrast': 1.05,
        'tta_bc_brightness': 0.02,
    },
    {
        'name': 'run_swinv2_base_192_focal',
        'seed': 42,
        'model_name': 'swinv2_base_window12_192',
        'img_size': 192,
        'batch_size': 12,
        'num_workers': 2,
        'num_workers_val': 2,
        'num_workers_test': 2,
        'epochs': 14,
        'lr': 1.0e-4,
        'weight_decay': 1e-4,
        'patience': 3,
        'n_splits': 5,
        'run_full_cv': True,
        'mixed_precision': True,
        'loss_name': 'focal',
        'label_smoothing': 0.08,
        'focal_gamma': 2.0,
        'use_class_weights': True,
        'use_tta': True,
        'tta_hflip': True,
        'tta_light_bc': False,
        'tta_bc_contrast': 1.05,
        'tta_bc_brightness': 0.02,
    },
]

def apply_recommended_config(index: int):
    if index < 1 or index > len(RECOMMENDED_RUN_CONFIGS):
        raise ValueError(f'index harus 1..{len(RECOMMENDED_RUN_CONFIGS)}')

    pick = RECOMMENDED_RUN_CONFIGS[index - 1]

    cfg.seed = int(pick['seed'])
    cfg.model_name = pick['model_name']
    cfg.img_size = int(pick['img_size'])
    cfg.batch_size = int(pick['batch_size'])
    cfg.num_workers = int(pick['num_workers'])
    cfg.num_workers_val = int(pick['num_workers_val'])
    cfg.num_workers_test = int(pick['num_workers_test'])

    cfg.epochs = int(pick['epochs'])
    cfg.lr = float(pick['lr'])
    cfg.weight_decay = float(pick['weight_decay'])
    cfg.patience = int(pick['patience'])
    cfg.n_splits = int(pick['n_splits'])
    cfg.run_full_cv = bool(pick['run_full_cv'])
    cfg.mixed_precision = bool(pick['mixed_precision'])

    cfg.loss_name = pick['loss_name']
    cfg.label_smoothing = float(pick['label_smoothing'])
    cfg.focal_gamma = float(pick['focal_gamma'])
    cfg.use_class_weights = bool(pick['use_class_weights'])

    cfg.use_tta = bool(pick['use_tta'])
    cfg.tta_hflip = bool(pick['tta_hflip'])
    cfg.tta_light_bc = bool(pick['tta_light_bc'])
    cfg.tta_bc_contrast = float(pick['tta_bc_contrast'])
    cfg.tta_bc_brightness = float(pick['tta_bc_brightness'])

    seed_everything(cfg.seed)

    global resolved_model_name
    resolved_model_name = resolve_timm_model_name(cfg.model_name)

    m = re.search(r'_(\d{3})(?:\.|$)', resolved_model_name)
    if m is not None:
        fixed_size = int(m.group(1))
        cfg.img_size = fixed_size

    print('Applied config:', pick['name'])
    print('resolved_model_name:', resolved_model_name)
    print('cfg:', asdict(cfg))

display(pd.DataFrame(RECOMMENDED_RUN_CONFIGS))
print('Gunakan apply_recommended_config(1..5), lalu jalankan cell Training.')

In [ ]:
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def softmax_np(x: np.ndarray) -> np.ndarray:
    z = x - np.max(x, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

def resolve_image_path(raw_path: str, label: str | None, is_test: bool) -> Path:
    p = Path(str(raw_path))
    if p.exists():
        return p

    name = p.name
    if is_test:
        cand = TEST_DIR / name
        if cand.exists():
            return cand
    else:
        if label is not None:
            cand = TRAIN_DIR / str(label) / name
            if cand.exists():
                return cand

    raise FileNotFoundError(f"File tidak ditemukan untuk path: {raw_path}")

split_run_df = split_df.copy()
split_run_df['resolved_path'] = [
    str(resolve_image_path(fp, lb, is_test=False))
    for fp, lb in zip(split_run_df['file_path'].tolist(), split_run_df['label'].tolist())
]

label_names = sorted(split_run_df['label'].unique().tolist())
label2idx = {lb: i for i, lb in enumerate(label_names)}
idx2label = {i: lb for lb, i in label2idx.items()}
split_run_df['label_idx'] = split_run_df['label'].map(label2idx)

if split_run_df['label_idx'].isna().any():
    raise RuntimeError('Ada label yang gagal dimapping ke index.')

test_run_df = test_meta_df.copy()
test_run_df['resolved_path'] = [
    str(resolve_image_path(fp, label=None, is_test=True))
    for fp in test_run_df['file_path'].tolist()
]

train_tfms = A.Compose([
    A.Resize(cfg.img_size, cfg.img_size),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.35),
    A.GaussNoise(p=0.2),
    A.ImageCompression(quality_range=(60, 100), p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

valid_tfms = A.Compose([
    A.Resize(cfg.img_size, cfg.img_size),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

class ImageClsDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: A.Compose, is_test: bool):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        path = row['resolved_path']
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f'Gagal baca image: {path}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        x = self.transform(image=img)['image']

        if self.is_test:
            return x, Path(path).name

        return x, int(row['label_idx']), int(row.name)

def make_loader(df: pd.DataFrame, transform: A.Compose, is_test: bool, shuffle: bool):
    ds = ImageClsDataset(df=df, transform=transform, is_test=is_test)
    workers = cfg.num_workers_test if is_test else (cfg.num_workers if shuffle else cfg.num_workers_val)
    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=workers,
        pin_memory=(DEVICE.type == 'cuda'),
        drop_last=False,
    )

print('Prepared training/test dataframe for runtime.')
print('train rows:', len(split_run_df), '| test rows:', len(test_run_df), '| classes:', len(label_names))

In [ ]:
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

class WeightedFocalLoss(nn.Module):
    def __init__(self, class_weights: torch.Tensor | None, gamma: float = 2.0):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, target: torch.Tensor):
        ce = nn.functional.cross_entropy(logits, target, reduction='none', weight=self.class_weights)
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

def build_model(num_classes: int):
    model = timm.create_model(resolved_model_name, pretrained=True, num_classes=num_classes)
    return model.to(DEVICE)

def build_criterion(class_weights: torch.Tensor | None):
    if cfg.loss_name == 'ce':
        return nn.CrossEntropyLoss(weight=class_weights)
    if cfg.loss_name == 'ce_ls':
        return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=cfg.label_smoothing)
    if cfg.loss_name == 'focal':
        return WeightedFocalLoss(class_weights=class_weights, gamma=cfg.focal_gamma)
    raise ValueError(f'Unsupported loss_name: {cfg.loss_name}')

def tta_views(images: torch.Tensor):
    views = [images]
    if cfg.tta_hflip:
        views.append(torch.flip(images, dims=[3]))
    if cfg.tta_light_bc:
        bc = images * cfg.tta_bc_contrast + cfg.tta_bc_brightness
        bc = torch.clamp(bc, -3.0, 3.0)
        views.append(bc)
        if cfg.tta_hflip:
            views.append(torch.flip(bc, dims=[3]))
    return views

def autocast_context():
    return torch.amp.autocast(device_type='cuda', enabled=cfg.mixed_precision and DEVICE.type == 'cuda')

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    running = 0.0

    for images, targets, _ in tqdm(loader, total=len(loader), desc='train', leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += float(loss.item()) * images.size(0)

    return running / max(len(loader.dataset), 1)

@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()
    running = 0.0
    logits_all, y_all, rowid_all = [], [], []

    for images, targets, row_ids in tqdm(loader, total=len(loader), desc='valid', leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        with autocast_context():
            logits = model(images)
            loss = criterion(logits, targets)

        running += float(loss.item()) * images.size(0)
        logits_all.append(logits.detach().float().cpu().numpy())
        y_all.extend(targets.detach().cpu().numpy().tolist())
        rowid_all.extend(row_ids.detach().cpu().numpy().tolist())

    val_logits = np.concatenate(logits_all, axis=0)
    return running / max(len(loader.dataset), 1), np.array(y_all), val_logits, np.array(rowid_all)

@torch.no_grad()
def predict_test_logits(model, loader, use_tta: bool):
    model.eval()
    logits_all = []
    file_all = []

    for images, file_names in tqdm(loader, total=len(loader), desc='test', leave=False):
        images = images.to(DEVICE, non_blocking=True)

        if use_tta:
            views = tta_views(images)
            logits_acc = None
            for view in views:
                with autocast_context():
                    logits_view = model(view)
                logits_acc = logits_view if logits_acc is None else (logits_acc + logits_view)
            logits = (logits_acc / len(views)).detach().float().cpu().numpy()
        else:
            with autocast_context():
                logits = model(images).detach().float().cpu().numpy()

        logits_all.append(logits)
        file_all.extend(list(file_names))

    return np.concatenate(logits_all, axis=0), file_all

print('Training utilities ready.')

In [ ]:
import time

run_name = f"colab_{resolved_model_name.replace('.', '_')}_img{cfg.img_size}_{time.strftime('%Y%m%d_%H%M%S')}"
run_ckpt_dir = CKPT_DIR / run_name
run_ckpt_dir.mkdir(parents=True, exist_ok=True)

folds_to_run = list(range(cfg.n_splits)) if cfg.run_full_cv else [0]
num_classes = len(label_names)

if 'row_id' not in split_run_df.columns:
    split_run_df = split_run_df.reset_index(drop=False).rename(columns={'index': 'row_id'})

oof_logits = np.full((len(split_run_df), num_classes), np.nan, dtype=np.float32)
test_logits_sum = np.zeros((len(test_run_df), num_classes), dtype=np.float32)
fold_scores = []

y_all_idx = split_run_df['label_idx'].values.astype(np.int64)
class_counts = np.bincount(y_all_idx, minlength=num_classes).astype(np.float32)
class_weights_np = class_counts.sum() / np.maximum(class_counts, 1.0)
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=DEVICE) if cfg.use_class_weights else None

print('run_name:', run_name)
print('folds_to_run:', folds_to_run)

for fold in folds_to_run:
    print('=' * 90)
    print(f'Fold {fold}')

    trn_df = split_run_df[split_run_df['fold'] != fold].copy().reset_index(drop=True)
    val_df = split_run_df[split_run_df['fold'] == fold].copy().reset_index(drop=True)

    trn_loader = make_loader(trn_df, train_tfms, is_test=False, shuffle=True)
    val_loader = make_loader(val_df, valid_tfms, is_test=False, shuffle=False)
    test_loader = make_loader(test_run_df, valid_tfms, is_test=True, shuffle=False)

    model = build_model(num_classes=num_classes)
    criterion = build_criterion(class_weights)
    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=max(cfg.epochs, 1), eta_min=cfg.lr * 0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=cfg.mixed_precision and DEVICE.type == 'cuda')

    best_f1 = -1.0
    best_epoch = -1
    patience_left = cfg.patience
    best_path = run_ckpt_dir / f'fold{fold}_best.pt'

    for epoch in range(1, cfg.epochs + 1):
        trn_loss = train_one_epoch(model, trn_loader, criterion, optimizer, scaler)
        val_loss, val_y, val_logits, val_rowids = eval_one_epoch(model, val_loader, criterion)
        val_pred = val_logits.argmax(axis=1)
        val_f1 = float(f1_score(val_y, val_pred, average='macro'))

        print(
            f"Epoch {epoch:02d} | trn_loss={trn_loss:.4f} | val_loss={val_loss:.4f} | val_macro_f1={val_f1:.4f}"
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch
            patience_left = cfg.patience
            torch.save(
                {
                    'model_state': model.state_dict(),
                    'fold': fold,
                    'best_epoch': best_epoch,
                    'best_f1': best_f1,
                    'model_name': resolved_model_name,
                    'num_classes': num_classes,
                    'label_names': label_names,
                    'cfg': asdict(cfg),
                },
                best_path,
            )
        else:
            patience_left -= 1
            if patience_left <= 0:
                print('Early stopping triggered')
                break

        scheduler.step()

    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])

    _, val_y, val_logits, val_rowids = eval_one_epoch(model, val_loader, criterion)
    val_pred = val_logits.argmax(axis=1)
    val_f1 = float(f1_score(val_y, val_pred, average='macro'))

    # Map validation logits back to global row_id
    val_global_rowids = val_df.iloc[val_rowids]['row_id'].values.astype(np.int64)
    oof_logits[val_global_rowids] = val_logits.astype(np.float32)

    fold_scores.append({'fold': fold, 'best_epoch': int(best_epoch), 'macro_f1': val_f1})

    test_logits, test_names = predict_test_logits(model, test_loader, use_tta=cfg.use_tta)
    test_logits_sum += test_logits.astype(np.float32)

    print(f'Fold {fold} done | best_epoch={best_epoch} | macro_f1={val_f1:.4f}')

# Aggregate predictions
test_logits_avg = test_logits_sum / max(len(folds_to_run), 1)
oof_probs = softmax_np(oof_logits)
test_probs = softmax_np(test_logits_avg)

# OOF score
oof_pred_idx = oof_probs.argmax(axis=1)
oof_macro_f1 = float(f1_score(y_all_idx, oof_pred_idx, average='macro'))
print(f'OOF macro F1: {oof_macro_f1:.6f}')

# Save fold scores
fold_scores_df = pd.DataFrame(fold_scores)
fold_scores_path = LOG_DIR / f'fold_scores_{run_name}.csv'
fold_scores_df.to_csv(fold_scores_path, index=False)
print('Saved fold scores:', fold_scores_path)

# Save OOF CSV
prob_cols = [f'prob_{idx2label[i]}' for i in range(num_classes)]
oof_out = split_run_df[['row_id', 'file_path', 'label', 'fold']].copy()
oof_out['pred_label'] = [idx2label[i] for i in oof_pred_idx]
oof_out[prob_cols] = oof_probs
oof_path = OOF_DIR / f'oof_{run_name}.csv'
oof_out.to_csv(oof_path, index=False)
print('Saved OOF:', oof_path)

# Save test probabilities and submission
test_prob_df = pd.DataFrame(test_probs, columns=prob_cols)
test_prob_df['id'] = [Path(p).stem for p in test_run_df['resolved_path']]
test_prob_df = sample_sub[['id']].merge(test_prob_df, on='id', how='left')
if test_prob_df[prob_cols].isna().any().any():
    raise RuntimeError('Ada probabilitas test yang missing setelah align ke sample submission.')

test_prob_path = TEST_PROB_DIR / f'test_probs_{run_name}.csv'
test_prob_df.to_csv(test_prob_path, index=False)
print('Saved test_probs:', test_prob_path)

sub_df = sample_sub[['id']].copy()
sub_df['label'] = [idx2label[i] for i in test_probs.argmax(axis=1)]
sub_path = SUB_DIR / f'submission_{run_name}.csv'
sub_df.to_csv(sub_path, index=False)
print('Saved submission:', sub_path)

# Save config used
cfg_out = asdict(cfg)
cfg_out['model_name_resolved'] = resolved_model_name
cfg_out['run_name'] = run_name
cfg_path = run_ckpt_dir / 'config_used.json'
cfg_path.write_text(json.dumps(cfg_out, indent=2))
print('Saved config:', cfg_path)

In [ ]:
from sklearn.metrics import log_loss

def softmax_temp(logits: np.ndarray, temp: float) -> np.ndarray:
    z = logits / max(temp, 1e-6)
    return softmax_np(z)

# Optional calibrated submission from OOF logits
if np.isnan(oof_logits).any():
    print('Skip calibration: oof_logits masih ada NaN.')
else:
    temp_grid = np.round(np.arange(0.7, 1.61, 0.05), 2)
    y_true = split_run_df['label_idx'].values.astype(np.int64)

    best_temp = 1.0
    best_nll = float('inf')
    for t in temp_grid:
        p = softmax_temp(oof_logits, float(t))
        nll = float(log_loss(y_true, p, labels=list(range(len(label_names)))))
        if nll < best_nll:
            best_nll = nll
            best_temp = float(t)

    print(f'Best temperature by OOF NLL: {best_temp} | NLL={best_nll:.6f}')

    test_probs_cal = softmax_temp(test_logits_avg, best_temp)
    sub_cal = sample_sub[['id']].copy()
    sub_cal['label'] = [idx2label[i] for i in test_probs_cal.argmax(axis=1)]

    cal_name = f'{run_name}_temp{str(best_temp).replace('.', 'p')}'
    sub_cal_path = SUB_DIR / f'submission_{cal_name}.csv'
    sub_cal.to_csv(sub_cal_path, index=False)
    print('Saved calibrated submission:', sub_cal_path)